**IN:** directory of pdfs we want to chunk and vectorize

**OUT:** chromadb vector databases for each paper, named by their ID

format:
- chromadb
  - BO1005
    - [collection "BO1005"]
  - JO1013
    - [collection "JO1013"]
  - ...



In [ ]:
# Import libraries
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
import os 

In [ ]:
# loop through all pdfs
pdf_files = [f for f in os.listdir("pdfs") if f.endswith('.pdf')]

for pdf_file in pdf_files:
    # paths
    PAPER_ID = pdf_file.split('-')[0]  # Extract ID from filename
    PDF_PATH = f"pdfs/{pdf_file}"
    CHROMA_PATH = f"chroma_db/{PAPER_ID}"

    # chromadb collection name
    chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
    collection = chroma_client.get_or_create_collection(name=PAPER_ID)
    
    # loading the document
    loader = PyPDFLoader(PDF_PATH)
    raw_documents = loader.load()

    # chunking the document
    # TODO see if we should edit these parameters and if so test out different options
    text_splitter = RecursiveCharacterTextSplitter( 
        chunk_size=500,
        chunk_overlap=100,
        length_function=len,
        is_separator_regex=False,
        # TODO add explicit embedding function (https://docs.trychroma.com/docs/embeddings/embedding-functions)
    )
    chunks = text_splitter.split_documents(raw_documents)
    
    documents = []
    ids = []
    # metadata = [] # I don't know if we need metadata

    i = 0
    for chunk in chunks:
        documents.append(chunk.page_content)
        ids.append("ID"+str(i))
        # metadata.append(chunk.metadata)
        i += 1

    # TODO add custom table chunks to collection
    
    # Adding to chromadb

    collection.upsert(
        documents=documents,
        ids=ids
        # metadatas=metadata,
    )


ValueError: Non-empty lists are required for ['ids', 'documents'] in upsert.

In [ ]:
# TODO 
# get chunks of specific tables in a better format such as a md table